# E2.9 · Regulator and auditor conversations

**Function E — AI for GRC → The Regulatory & Compliance Lead**  ·  *Security of AI*

Builds on **[E2.8 · Auditability of autonomous action](https://spbreed.github.io/cyber-commons/lessons/E2.8.html)**.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

A supervisor can tell the difference between confidence and evidence. The hardest part of this conversation is framing genuine uncertainty without sounding like you have lost control of the estate.

## 2 · The framework

```
   what a supervisor hears

   "we are confident"          -> on what evidence?
   "we test continuously"      -> show me a failure you caught
   "we do not know yet, and    -> credible
    here is how we will know"

   framing uncertainty without surrendering the room
```

Conversations with regulators and auditors go well when you bring the number
that is weakest and explain it, and badly when you bring the strongest and let
them find the other one.

That is not a moral point, it is a practical one. A supervisor who discovers a
weakness you did not disclose now doubts everything else you said, and the rest
of the engagement is spent re-establishing credibility you had at the start.

Three things to bring, in this order:

1. **The distinction you understand.** Conformance versus accuracy (B2.11).
   Volunteering it demonstrates you know what your own numbers mean.
2. **Your current coverage, honestly stated**, including stale and unevidenced
   controls (E1.7).
3. **The control you have not deployed, and the date you will.**

The third one is the one people omit, and it is the one that most reliably
converts scepticism into a working relationship.

## 3 · Demo — produce the two numbers, then the coverage

In [ ]:
import json, time
from dataclasses import dataclass
now = time.time(); DAY = 86400

@dataclass
class Truth:
    qid: str; cwe: str; file: str

def path_key(p):
    parts = [x for x in p.replace("\\","/").split("/") if x not in ("",".")]
    return "/".join(parts[-2:]) if len(parts) > 1 else (parts[-1] if parts else "")

TRUTHS = {f"q{i}": Truth(f"q{i}", ["CWE-89","CWE-78"][i % 2],
                         f"{['CWE-89','CWE-78'][i % 2]}/{i}.py") for i in range(1, 21)}
ANSWERS = {q: json.dumps({"qid": q, "cwe": "CWE-89", "file": t.file,
                          "rationale": "untrusted input is concatenated"})
           for q, t in TRUTHS.items()}

def evaluate(answers, truths):
    conf = expert = 0
    for q, t in truths.items():
        try: d = json.loads(answers[q])
        except (json.JSONDecodeError, KeyError): continue
        conf += 1
        if path_key(d["file"]) != path_key(t.file): continue
        expert += 1.0 if d["cwe"].upper() == t.cwe else 0.5
    return {"n": len(truths), "conformance": round(conf/len(truths), 4),
            "expert_accuracy": round(expert/len(truths), 4)}

R = evaluate(ANSWERS, TRUTHS)
print(f"n                {R['n']}")
print(f"conformance      {R['conformance']:.4f}")
print(f"expert accuracy  {R['expert_accuracy']:.4f}")

In [ ]:
@dataclass
class ControlTest:
    cid: str; passed: bool; tested_at: float; valid_for_days: float
    def state(self, at):
        if (at - self.tested_at)/DAY > self.valid_for_days: return "STALE"
        return "PASS" if self.passed else "FAIL"

REQUIRED = ["AC-1","AC-2","SB-1","SB-2","EV-1","EV-2","DR-1","ST-1"]
TESTS = {t.cid: t for t in [
 ControlTest("AC-1", True,  now -  4*DAY, 30),
 ControlTest("AC-2", True,  now -  9*DAY, 30),
 ControlTest("SB-1", True,  now - 45*DAY, 30),
 ControlTest("EV-1", True,  now -  5*DAY, 60),
 ControlTest("EV-2", True,  now - 12*DAY, 30)]}

rows = [(c, TESTS[c].state(now) if c in TESTS else "NO EVIDENCE") for c in REQUIRED]
evidenced = sum(1 for _, s in rows if s == "PASS")
print(f"{'control':9s}{'state':14s}")
print("-" * 24)
for c, s in rows: print(f"{c:9s}{s:14s}")
print(f"\ncoverage: {evidenced}/{len(REQUIRED)} = {evidenced/len(REQUIRED):.0%}")

## 4 · Where it breaks — leading with the strongest number

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">how you open</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">what it is</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">what happens next</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">“Our harness scores 100%.”</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">conformance quoted as quality</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>the follow-up “against what key?” ends the meeting's credibility</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">“We have full coverage of our AI controls.”</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">stale and untested counted as passing</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>one request for dates exposes it</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">“Conformance is 100%; expert accuracy is 50% against a held-out key of 20. Coverage is 63%, with SB-1 stale at 45 days and two controls not yet deployed.”</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">both numbers, honestly</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">nothing left for them to discover — the conversation moves to the plan</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">The third opening is the only one that survives a follow-up, and it is the one that sounds worst when you rehearse it.</div>

## 5 · The control — the disclosure script, generated from live state

In [ ]:
def disclosure(evalr, rows, required):
    evidenced = [c for c, s in rows if s == "PASS"]
    stale     = [c for c, s in rows if s == "STALE"]
    missing   = [c for c, s in rows if s == "NO EVIDENCE"]
    failing   = [c for c, s in rows if s == "FAIL"]
    plan = {"SB-1": "re-tested by 2026-08-31 (automation in progress)",
            "DR-1": "drift alerting deployed by 2026-10-15",
            "ST-1": "game day scheduled 2026-09-12"}
    lines = [
      f"1. Two numbers, and they measure different things.",
      f"   conformance {evalr['conformance']:.0%} — schema validity, structural, "
      f"not a quality claim.",
      f"   expert accuracy {evalr['expert_accuracy']:.0%} against a held-out key "
      f"of {evalr['n']} questions. That is the number that means something.",
      f"",
      f"2. Control coverage {len(evidenced)}/{len(required)} = "
      f"{len(evidenced)/len(required):.0%}, counting only controls that are "
      f"currently evidenced.",
      f"   stale: {stale or 'none'}   failing: {failing or 'none'}   "
      f"no evidence: {missing or 'none'}",
      f"",
      f"3. What we have not done, and when we will:",
    ]
    for c in stale + failing + missing:
        lines.append(f"   {c}: {plan.get(c, 'plan to be confirmed')}")
    return "\n".join(lines)

print(disclosure(R, rows, REQUIRED))
print("\nRehearse the second section out loud. If naming your weakest control is")
print("uncomfortable, that discomfort is the reason to say it first.")
assert R["expert_accuracy"] < R["conformance"]

## What you just proved

Conformance is 1.0000 while expert accuracy is 0.5000 on a 20-question held-out key. Coverage is 5 of 8 with SB-1 stale and DR-1 and ST-1 unevidenced. The three openings show conformance-as-quality and inflated coverage failing on the first follow-up, and the generated disclosure script states both numbers, the stale and missing controls, and a date for each.

## Your turn

Generate this script from your own live control state rather than writing it. If it cannot be generated, your coverage number is being assembled by hand for each meeting — which is why it differs between meetings.

## Where this leaves you

**What you can do now.** One control set mapped to a horizontal regime, a sector overlay and a privacy position; trigger criteria for disclosure written before they are needed; documentation and logs designed backwards from what a supervisor will ask.

**What you still cannot do.** Compliance tells you what you owe. It does not tell you what to build first, who owns it, how to say no to the business without losing the next conversation, or what to do when the programme is judged on a failure it was never going to prevent.

**Chapter 12 is the programme itself, run from the CISO office. Next → E3.1, translating agentic risk upward.**

---

**Next → [E3.1 · Translating agentic risk upward](https://spbreed.github.io/cyber-commons/lessons/E3.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*